# Create a generative AI app that uses your own data

## CLI Azure

Install and authenticate the Azure CLI:



In [ ]:
az login

Create a `.env` file:

In [ ]:
OPEN_AI_ENDPOINT=https://your-openai-resource.openai.azure.com/
OPEN_AI_KEY=your-openai-api-key
CHAT_MODEL=gpt-4o
EMBEDDING_MODEL=text-embedding-ada-002
SEARCH_ENDPOINT=https://your-search-service.search.windows.net
SEARCH_KEY=your-search-api-key
INDEX_NAME=brochures-index
AZURE_SUBSCRIPTION_ID=your-subscription-id
AZURE_RESOURCE_GROUP=rg-rag-openai-lab
AZURE_LOCATION=eastus2
AZURE_AI_HUB_NAME=ragapp-hub-sdk-cli
AZURE_AI_PROJECT_NAME=ragapp-pj-sdk-cli

### Create an Azure AI Foundry Hub and Project

Create a resource group:

In [ ]:
az group create --name rg-rag-openai-lab --location eastus2

Create an Azure AI Foundry hub:

In [ ]:
az ml workspace create --kind hub --resource-group rg-rag-openai-lab --name ragapp-hub-sdk-cli --display-name "RAG App Display"

Create an Azure AI Services resource for OpenAI:

In [ ]:
az ml workspace create --kind project --hub-id "/subscriptions/{subscription_id}/resourceGroups/rg-rag-openai-lab/providers/Microsoft.MachineLearningServices/workspaces/ragapp-hub-sdk-cli" --resource-group rg-rag-openai-lab --name ragapp-pj-sdk-cli --display-name "RAG App Display"

View project settings

In [ ]:
az ml workspace show --name ragapp-pj-sdk-cli --resource-group rg-rag-openai-lab

Create an Azure AI Services resource for OpenAI:

In [ ]:
az cognitiveservices account create --name ragapp-hub-sdk-cli --resource-group rg-rag-openai-lab --location eastus2 --kind OpenAI --sku S0

Retrieve the Azure AI Services endpoint and key:

In [ ]:
az cognitiveservices account show --name ragapp-hub-sdk-cli --resource-group rg-rag-openai-lab | jq -r .properties.endpoint

In [ ]:
az cognitiveservices account keys list --name ragapp-hub-sdk-cli --resource-group  rg-rag-openai-lab | jq -r .key1

### Deploy Models

List Models

In [ ]:
az cognitiveservices account list-models -n ragapp-hub-sdk-cli -g rg-rag-openai-lab | jq '.[] | { name: .name, format: .format, version: .version, sku: .skus[0].name, capacity: .skus[0].capacity.default }'

Deploy `text-embedding-ada-002` model:

In [ ]:
{
  "name": "text-embedding-ada-002",
  "format": "OpenAI",
  "version": "2",
  "sku": "Standard",
  "capacity": 120
}

In [ ]:
az cognitiveservices account deployment create --name ragapp-hub-sdk-cli --resource-group  rg-rag-openai-lab --deployment-name embedding-deployment --model-name text-embedding-ada-002 --model-version "2" --model-format OpenAI --sku-capacity "50" --sku-name "Standard"

Deploy `gpt-4.1` model:



In [ ]:
{
  "name": "gpt-4.1",
  "format": "OpenAI",
  "version": "2025-04-14",
  "sku": "GlobalStandard",
  "capacity": 10
}


In [ ]:
az cognitiveservices account deployment create --name ragapp-hub-sdk-cli --resource-group  rg-rag-openai-lab --deployment-name gpt41-employment --model-name gpt-4.1 --model-version "2025-04-14" --model-format OpenAI --sku-capacity "50" --sku-name "GlobalStandard"

### Add Data to Your Project

List Locations Avalible

In [ ]:
az account list-locations --query "[].{Region:name}" --out table

Create Storage Account (optional)

In [ ]:
az storage account create \
  --name ragstoragepdfs \
  --resource-group rg-rag-openai-lab \
  --location eastus2 \
  --sku Standard_LRS \
  --kind StorageV2


Create Container ([view documentation](https://learn.microsoft.com/en-us/azure/storage/blobs/blob-containers-cli)):

In [ ]:
az storage container create \
  --name brochures \
  --account-name ragstoragepdfs \
  --auth-mode login

Get Endpoint and apikeys

In [ ]:
az storage account show --resource-group rg-rag-openai-lab --name ragstoragepdfs --query '[primaryEndpoints, secondaryEndpoints]'

List maximum containers

In [ ]:
az storage container list \
    --account-name ragstoragepdfs \
    --auth-mode login

Upload multiples Files recursively, upload blobs ([View Documentation](https://learn.microsoft.com/en-us/azure/storage/blobs/blob-cli)):

In [ ]:
az storage blob upload-batch \
    --destination brochures \
    --source ./brochures \
    --account-name ragstoragepdfs \
    --auth-mode login \
    --pattern *.pdf

List all blobs in a container by name.

In [ ]:
az storage blob list \
    --account-name ragstoragepdfs \
    --container brochures \
    --query "[].name" \
    --auth-mode login \
    --output tsv

Download a single named blob:

In [ ]:
az storage blob download \
    --container brochures \
    --file "$destinationPath$destinationFilename" \
    --name "demo-file.txt" \
    --account-name ragstoragepdfs \
    --auth-mode login

Download multiple blobs using a pattern value

In [ ]:
az storage blob download-batch \
    --destination "./data" \
    --source brochures \
    --pattern images/*.png \
    --account-name ragstoragepdfs \
    --auth-mode key

Log data resource in your Foundry project

In [ ]:
az ml data create \
  --name brochures \
  --path https://ragstoragepdfs.blob.core.windows.net/brochures \
  --type uri_folder \
  --resource-group rg-rag-openai-lab \
  --workspace-name ragapp-pj-sdk-cli


Storage apikeys and endpoints

In [ ]:
az storage account keys list \
  --account-name ragstoragepdfs \
  --resource-group rg-rag-openai-lab


### Prepare the application configuration

OPEN_AI_ENDPOINT

output: `https://rag-openai-hub-aiservices.openai.azure.com/`

In [ ]:
az cognitiveservices account show \
  --name rag-openai-hub-aiservices \
  --resource-group rg-rag-lab \
  --query "properties.endpoint" \
  --output tsv


### Create an index for your data

Create Azure AI Search services

In [ ]:
az search service create \
  --name ragsearchsvc \
  --resource-group rg-rag-openai-lab \
  --location eastus2 \
  --sku Basic

OPEN_AI_KEY

you can use key1 or key2

In [ ]:
az cognitiveservices account keys list \
  --name rag-openai-hub-aiservices \
  --resource-group rg-rag-lab \
  --query "key1" \
  --output tsv


SEARCH_ENDPOINT

🔹 output: ragsearchsvc.search.windows.net

🔹 for `.env`, use: https://ragsearchsvc.search.windows.net

In [ ]:
az search service show \
  --name ragsearchsvc \
  --resource-group rg-rag-lab \
  --query "properties.hostName" \
  --output tsv


SEARCH_KEY

In [ ]:
az search admin-key show \
  --service-name ragsearchsvc \
  --resource-group rg-rag-lab \
  --query "primaryKey" \
  --output tsv

CHAT_MODEL and EMBEDDING_MODEL

These values are defined by you at the time of deploying the models:

In [ ]:
az cognitiveservices account deployment list \
  --name rag-openai-hub-aiservices \
  --resource-group rg-rag-lab \
  --query "[].{name:name, model:model.name}" \
  --output table


Search the deployment names for:

* gpt-4.1 → example: gpt41-deployment

* text-embedding-ada-002 → example: embedding-deployment

Use those names as values for:

In [ ]:
CHAT_MODEL=gpt41-deployment
EMBEDDING_MODEL=embedding-deployment

Create Vectorial Index

In [ ]:
{
  "name": "brochures-index",
  "fields": [
    { "name": "id", "type": "Edm.String", "key": true, "filterable": true },
    { "name": "content", "type": "Edm.String", "searchable": true },
    { "name": "contentVector", "type": "Collection(Edm.Single)", "searchable": true, "vectorSearchConfiguration": "vector-config" }
  ],
  "vectorSearch": {
    "algorithmConfigurations": [
      {
        "name": "vector-config",
        "kind": "hnsw",
        "parameters": {
          "m": 4,
          "efConstruction": 400,
          "efSearch": 500,
          "metric": "cosine"
        }
      }
    ]
  }
}


In [ ]:
az search index create \
  --service-name ragsearchsvc \
  --resource-group rg-rag-openai-lab \
  --name brochures-index \
  --definition @index-definition.json


Embedding + chunking

In [ ]:
pip install azure-search-documents openai python-dotenv PyMuPDF

In [ ]:
OPENAI_API_KEY=your-openai-key
OPENAI_ENDPOINT=https://your-openai-resource.openai.azure.com/
SEARCH_API_KEY=your-search-key
SEARCH_ENDPOINT=https://ragsearchsvc.search.windows.net
INDEX_NAME=brochures-index
EMBEDDING_MODEL=text-embedding-ada-002

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

OPENAI_API_KEY = os.getenv("OPEN_AI_KEY")
OPENAI_ENDPOINT = os.getenv("OPEN_AI_ENDPOINT")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL")
SEARCH_API_KEY = os.getenv("SEARCH_KEY")
SEARCH_ENDPOINT = os.getenv("SEARCH_ENDPOINT")
INDEX_NAME = os.getenv("INDEX_NAME")


Crack: extract text from PDFs

In [ ]:
import fitz  # PyMuPDF
import os

def extract_text_from_pdfs(folder_path):
    documents = []
    for filename in os.listdir(folder_path):
        if filename.endswith(".pdf"):
            doc = fitz.open(os.path.join(folder_path, filename))
            text = ""
            for page in doc:
                text += page.get_text()
            documents.append({"id": filename, "content": text})
    return documents


Chunking

In [ ]:
def chunk_text(documents, chunk_size=500):
    chunks = []
    for doc in documents:
        text = doc["content"]
        for i in range(0, len(text), chunk_size):
            chunk = text[i:i+chunk_size]
            chunks.append({
                "id": f"{doc['id']}_{i}",
                "content": chunk
            })
    return chunks


Upload vector index to Azure AI Search

In [ ]:
from azure.search.documents import SearchClient
from azure.core.credentials import AzureKeyCredential

search_client = SearchClient(
    endpoint=os.getenv("SEARCH_ENDPOINT"),
    index_name=os.getenv("INDEX_NAME"),
    credential=AzureKeyCredential(os.getenv("SEARCH_API_KEY"))
)

def upload_to_search(chunks):
    batch = []
    for chunk in chunks:
        batch.append({
            "id": chunk["id"],
            "content": chunk["content"],
            "contentVector": chunk["contentVector"]
        })
    result = search_client.upload_documents(documents=batch)
    print("Upload result:", result)


Generate embeddings with Azure OpenAI

In [ ]:
import openai
from dotenv import load_dotenv

load_dotenv()
openai.api_key = os.getenv("OPENAI_API_KEY")
openai.api_base = os.getenv("OPENAI_ENDPOINT")
openai.api_type = "azure"
openai.api_version = "2023-05-15"

def embed_chunks(chunks):
    for chunk in chunks:
        response = openai.Embedding.create(
            input=chunk["content"],
            engine=os.getenv("EMBEDDING_MODEL")
        )
        chunk["contentVector"] = response["data"][0]["embedding"]
    return chunks


full flow
```bash
docs = extract_text_from_pdfs("./brochures")
chunks = chunk_text(docs)
embedded_chunks = embed_chunks(chunks)
upload_to_search(embedded_chunks)

```

## SDK Python